# SignSense AI — MobileNetV3 CNN Fine-Tune Notebook

**Model:** MobileNetV3Small transfer learning on 224×224 hand crop images  
**Input:** 224×224×3 RGB hand crop  
**Architecture:** MobileNetV3Small (frozen) → Dropout → Dense(29, softmax)  
**Target accuracy:** > 90%  
**Runtime:** ~2h on Colab T4 GPU (two-phase training)

---
### Two-phase training strategy
- **Phase 1** (~20 epochs): Frozen MobileNetV3 base, train head only. Fast convergence.
- **Phase 2** (~40 epochs): Unfreeze top layers from index 80, fine-tune with very low LR (1e-5).

### Prerequisites
- Run `train_mlp.ipynb` first — it downloads the dataset and runs `preprocess.py --all`
- `preprocess.py` saves image crops to `data/processed/ASL/crops/<class>/`
- This notebook requires those crops to exist

### Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_MODELS_DIR = '/content/drive/MyDrive/SignSense/models'
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
print(f'Drive mounted. Models will be saved to: {DRIVE_MODELS_DIR}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q mediapipe==0.10.14 scikit-learn tqdm
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Upload backend code to Colab ────────────────────────────────────
# Step 1: Run this PowerShell script LOCALLY to create the zip:
#   .\notebooks\create_colab_zip.ps1
#
# Step 2: Upload backend_colab.zip using the button below
#   (Files panel on the left → Upload, OR run the cell to get a file picker)

from google.colab import files
import os, sys, zipfile

BACKEND_PATH = '/content/backend'

if not os.path.exists(BACKEND_PATH):
    print('Upload backend_colab.zip when the file picker appears...')
    uploaded = files.upload()  # opens file picker
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content')
    print(f'Extracted {zip_name} to /content/')
else:
    print('backend/ already exists, skipping upload.')

# Add backend to Python path
sys.path.insert(0, BACKEND_PATH)

# Verify
from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'Import OK — {NUM_CLASSES} classes: {ASL_CLASSES[:5]}...')

In [ ]:
# ── Cell 4: Verify image crops exist ─────────────────────────────────────────
import os
from pathlib import Path

CROPS_DIR = '/content/backend/data/processed/ASL/crops'

if not os.path.exists(CROPS_DIR):
    raise FileNotFoundError(
        f'Image crops not found at {CROPS_DIR}\n'
        'Run train_mlp.ipynb first — preprocess.py saves crops automatically.'
    )

# Count crops per class
class_dirs = sorted(Path(CROPS_DIR).iterdir())
total = 0
print(f'Found {len(class_dirs)} class directories:')
for d in class_dirs:
    count = len(list(d.glob('*.jpg')))
    total += count
    print(f'  {d.name:<10} {count:>5} crops')
print(f'\nTotal crops: {total}')

In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU memory growth enabled.')
else:
    print('WARNING: No GPU. CNN fine-tuning will take 8-12h on CPU. Not recommended.')

In [ ]:
# ── Cell 6: Train MobileNetV3 (Phase 1 + Phase 2) ────────────────────────────
from pathlib import Path
from configs.training_config import CNNConfig
from src.train import train_cnn

cfg = CNNConfig()
cfg.save_dir = Path(DRIVE_MODELS_DIR)
cfg.log_dir  = Path('/content/logs/cnn')
cfg.mixed_precision = True  # Safe for CNN on T4

print('Config:')
print(f'  base_model:      {cfg.base_model}')
print(f'  input_shape:     {cfg.input_shape}')
print(f'  fine_tune_from:  {cfg.fine_tune_from}')
print(f'  Phase 1 epochs:  {cfg.phase1_epochs}  lr={cfg.phase1_lr}')
print(f'  Phase 2 epochs:  {cfg.phase2_epochs}  lr={cfg.phase2_lr}')
print(f'  mixed_precision: {cfg.mixed_precision}')
print()

model = train_cnn(cfg)
if model is None:
    print('ERROR: CNN training failed. Check that image crops exist.')
else:
    print('\nTraining complete!')

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
# CNN evaluate() uses landmark data (not image crops) for simplicity.
# For image-based evaluation, use the tf.data pipeline directly.
from pathlib import Path
import configs.training_config as tc
tc.MODELS_DIR = Path(DRIVE_MODELS_DIR)

from src.evaluate import evaluate
results = evaluate('asl_mobilenet', split='test')
print(f"\nFinal test accuracy: {results['accuracy']*100:.1f}%")
print(f"Top-5 accuracy:      {results['top5']*100:.1f}%")

In [ ]:
# ── Cell 8: TensorBoard ───────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/logs/cnn

In [ ]:
# ── Cell 9: Verify saved model ────────────────────────────────────────────────
import os, numpy as np, tensorflow as tf

saved_files = os.listdir(DRIVE_MODELS_DIR)
print('Files saved to Drive:')
for f in saved_files:
    size_mb = os.path.getsize(os.path.join(DRIVE_MODELS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

# Sanity check: load and run one prediction
loaded = tf.keras.models.load_model(os.path.join(DRIVE_MODELS_DIR, 'asl_mobilenet.keras'))
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
pred = loaded.predict(dummy, verbose=0)
print(f'\nSanity check — output shape: {pred.shape}  sum: {pred.sum():.4f} (should be ~1.0)')

In [ ]:
# ── Cell 10: All models summary ───────────────────────────────────────────────
# Run this after all three notebooks to compare models side by side
from pathlib import Path
import configs.training_config as tc
tc.MODELS_DIR = Path(DRIVE_MODELS_DIR)

from src.evaluate import benchmark_all
benchmark_all(split='test')